<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# The pulse of NYC yellow-taxi pickups

Vector analytics at real-world volume. We answer:

> **Which NYC zones are busiest at which hour, and what are the dominant origin-destination flows?**

This notebook reads ~3 million rows of yellow-taxi trips from the public NYC TLC mirror, joins them to taxi-zone polygons, finds each zone's peak hour with a window function, derives the top origin→destination flows as `ST_MakeLine` geometries on zone centroids, bins zone activity into Bing tiles for a city-wide heatmap, writes the per-zone result as **GeoParquet 1.1**, and renders the whole thing in **SedonaKepler**.

**Data sources** — taxi-zone polygons ship with the docker image (NYC TLC, public). Trip data is fetched at runtime from the [TLC Cloudfront mirror](https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet).

<!-- requires-network: true -->

*This notebook requires outbound network access to download the trip parquet (~50 MB). Set `SEDONA_NOTEBOOK_OFFLINE=1` to skip it under sandboxed CI.*

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

In [ ]:
# Taxi-zone polygons ship in NAD83 / NY State Plane (feet). Reproject to
# WGS84 so we can join to lon/lat-derived geometries downstream and so
# Kepler renders them on a web-mercator basemap. ST_Transform is now
# backed by proj4sedona, which accepts CRS strings, EPSG codes,
# PROJJSON, and WKT2 alike.
zones_raw = sedona.read.format("shapefile").load("data/nyc_taxi_zones")
zones = zones_raw.selectExpr(
    "LocationID as location_id",
    "zone",
    "borough",
    "ST_Transform(geometry, 'epsg:2263', 'epsg:4326') as geometry",
)
zones.cache()
print(f"{zones.count()} taxi zones")
zones.show(5, truncate=40)

In [ ]:
# Pull one month of yellow-taxi trips. Spark can't currently read parquet
# directly from an https:// URL through its Hadoop FileSystem layer, so
# we cache the file to /tmp and point Spark at the local copy.
import os
import urllib.request

TRIP_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
)
TRIP_PATH = "/tmp/yellow_tripdata_2024-01.parquet"

if not os.path.exists(TRIP_PATH):
    print(f"downloading {TRIP_URL} ...")
    urllib.request.urlretrieve(TRIP_URL, TRIP_PATH)

trips = sedona.read.parquet(TRIP_PATH).selectExpr(
    "PULocationID as pu_zone",
    "DOLocationID as do_zone",
    "hour(tpep_pickup_datetime) as pickup_hour",
    "tpep_pickup_datetime as pickup_ts",
)
trips = trips.where("pickup_ts >= '2024-01-01' AND pickup_ts < '2024-02-01'")
print(f"{trips.count():,} trips")
trips.show(5)

In [ ]:
# Aggregate trips per (zone, hour) and label each zone with its peak hour
# bucket: morning (5-10), midday (11-15), evening (16-19), nightlife (20-4).
trips.createOrReplaceTempView("trips")

zone_hour = sedona.sql("""
    SELECT pu_zone, pickup_hour, COUNT(*) AS trips
    FROM trips
    GROUP BY pu_zone, pickup_hour
""")
zone_hour.createOrReplaceTempView("zone_hour")

zone_profile = sedona.sql("""
    WITH ranked AS (
        SELECT pu_zone, pickup_hour, trips,
               ROW_NUMBER() OVER (PARTITION BY pu_zone ORDER BY trips DESC) AS rn
        FROM zone_hour
    )
    SELECT pu_zone,
           SUM(trips)              AS total_trips,
           MAX_BY(pickup_hour, trips)  AS peak_hour,
           CASE
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 5  AND 10 THEN 'morning'
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 11 AND 15 THEN 'midday'
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 16 AND 19 THEN 'evening'
               ELSE                                                   'nightlife'
           END                     AS peak_bucket
    FROM zone_hour
    GROUP BY pu_zone
""")
zone_profile.show(10)

In [ ]:
# Attach zone geometries and bin each zone's centroid into a Bing tile
# quadkey for a city-wide density layer. ST_BingTileAt(lon, lat, zoom)
# is new in Sedona 1.9 and folds cleanly into a SQL projection.
zones.createOrReplaceTempView("zones")
zone_profile.createOrReplaceTempView("zone_profile")

zones_with_pulse = sedona.sql("""
    SELECT z.location_id, z.zone, z.borough,
           p.total_trips, p.peak_hour, p.peak_bucket,
           z.geometry,
           ST_Centroid(z.geometry)                                  AS centroid,
           ST_BingTileAt(ST_X(ST_Centroid(z.geometry)),
                         ST_Y(ST_Centroid(z.geometry)), 14)         AS tile_quadkey
    FROM zones z
    JOIN zone_profile p ON z.location_id = p.pu_zone
""")
zones_with_pulse.cache()
zones_with_pulse.select(
    "zone", "borough", "total_trips", "peak_hour", "peak_bucket", "tile_quadkey"
).show(10, truncate=30)

In [ ]:
# Top origin-destination flows as line geometries between zone centroids.
# ST_MakeLine + ST_Centroid is the idiomatic way to derive flow geometries
# from a zone-attributed OD matrix.
od = sedona.sql("""
    SELECT pu_zone, do_zone, COUNT(*) AS trips
    FROM trips
    WHERE pu_zone <> do_zone
    GROUP BY pu_zone, do_zone
    ORDER BY trips DESC
    LIMIT 30
""")
od.createOrReplaceTempView("od")

flows = sedona.sql("""
    SELECT od.pu_zone, od.do_zone, od.trips,
           pu.zone AS pu_name, do_.zone AS do_name,
           ST_MakeLine(ST_Centroid(pu.geometry),
                       ST_Centroid(do_.geometry)) AS flow_geom
    FROM od
    JOIN zones pu  ON pu.location_id  = od.pu_zone
    JOIN zones do_ ON do_.location_id = od.do_zone
""")
flows.select("pu_name", "do_name", "trips").show(10, truncate=30)

In [ ]:
# Persist per-zone aggregates as GeoParquet 1.1. Sedona auto-populates
# covering-bbox metadata and projjson CRS from the geometry SRID; the
# resulting file can be filter-pushdown-pruned on a bbox without scanning
# every row.
import shutil

out_path = "/tmp/nyc_taxi_zone_pulse.parquet"
if os.path.exists(out_path):
    shutil.rmtree(out_path)

(
    zones_with_pulse.select(
        "location_id",
        "zone",
        "borough",
        "total_trips",
        "peak_hour",
        "peak_bucket",
        "tile_id",
        "geometry",
    )
    .coalesce(1)
    .write.format("geoparquet")
    .save(out_path)
)

sedona.read.format("geoparquet").load(out_path).select(
    "zone", "borough", "total_trips", "peak_bucket"
).show(5, truncate=30)

In [ ]:
# Three-layer Kepler map: zones colored by peak-hour bucket, centroids
# scaled by trip count, top-30 OD flowlines.
from sedona.spark import SedonaKepler

zones_layer = sedona.read.format("geoparquet").load(out_path).drop("tile_id")
centroids_layer = zones_with_pulse.selectExpr(
    "zone", "borough", "total_trips", "peak_bucket", "centroid as geometry"
)
flows_layer = flows.selectExpr("pu_name", "do_name", "trips", "flow_geom as geometry")

kepler_map = SedonaKepler.create_map(df=zones_layer, name="zones_by_peak")
SedonaKepler.add_df(kepler_map, centroids_layer, name="zone_centroids")
SedonaKepler.add_df(kepler_map, flows_layer, name="top_od_flows")
kepler_map

## What just happened?

We turned 3 million raw trip records into a labelled spatial dataset in a handful of cells:

1. **Reprojected** taxi-zone polygons from NY State Plane to WGS84 with `ST_Transform`, now backed by proj4sedona.
2. **Aggregated** trips per `(zone, hour)` and used a window function to find each zone's peak hour and bucket it (morning / midday / evening / nightlife).
3. **Bing-tiled** zone centroids with the new `ST_BingTileXY` for an evenly-spaced density grid.
4. **Built flow geometries** for the top origin-destination pairs via `ST_MakeLine` over `ST_Centroid` outputs.
5. **Wrote** the per-zone result as **GeoParquet 1.1** with auto covering-bbox metadata, then read it back.
6. **Visualized** zones, centroids, and flowlines as three layers on a single SedonaKepler map.

Sedona ran the same SQL on a single laptop here that it would run across a thousand-node cluster on a year of TLC data. No code change needed — just more workers.